# Pipeline walkthrough — one image end-to-end

Demonstrates metadata extraction, gimbal projection, per-lamp state, and bbox derivation for a single drone frame. Useful for debugging changes to the projection module or the gimbal calibration.

In [ ]:
from pathlib import Path
import sys, yaml
REPO = Path.cwd().parents[0] if Path.cwd().name == 'notebooks' else Path.cwd()
sys.path.insert(0, str(REPO / 'src'))
from papi.metadata import extract_image_metadata
from papi.geometry import resolve_papi_for_frame
from papi.projection import ProjectionConvention, DEFAULT_CONVENTION, project_papi_lights, world_to_image
from papi.lamp_state import compute_lamp_state
from papi.global_state import derive_global_state

In [ ]:
airport_cfg = yaml.safe_load((REPO / 'configs' / 'papi_edny.yaml').read_text())
proj_path = REPO / 'configs' / 'projection.yaml'
convention = (
    ProjectionConvention.from_dict(yaml.safe_load(proj_path.read_text())['convention'])
    if proj_path.exists() else DEFAULT_CONVENTION
)
convention

In [ ]:
# Pick the first WideCamera frame in the first flight
raw = REPO / 'data' / 'raw'
first_jpg = next((p for p in sorted(raw.glob('*/*.JPG'))), None)
row = extract_image_metadata(first_jpg)
row

In [ ]:
# Pick the nearer PAPI runway for this frame and project the 4 lights into pixels
runway, papi_for_frame = resolve_papi_for_frame(row, airport_cfg)
cam_cfg = airport_cfg['cameras']['wide' if row['camera'] == 'WideCamera' else 'zoom']
projected = project_papi_lights(row, papi_for_frame, cam_cfg, convention)
print(f'Target runway: {runway}')
projected

In [ ]:
lamps, margin = compute_lamp_state(row, papi_for_frame)
lamps, derive_global_state(lamps), margin

In [ ]:
# LRF round-trip: project LRF target through the same pipeline; should land near (W/2, H/2)
if row['lrf_status'] == 'Normal':
    u, v, behind, in_frame = world_to_image(
        target_lat=row['lrf_target_lat'],
        target_lon=row['lrf_target_lon'],
        target_alt_m=row['lrf_target_abs_alt_m'] or 465.0,
        camera_lat=row['lat'], camera_lon=row['lon'], camera_alt_m=row['alt_ellipsoidal_m'],
        gimbal_yaw_deg=row['gimbal_yaw_deg'], gimbal_pitch_deg=row['gimbal_pitch_deg'], gimbal_roll_deg=row['gimbal_roll_deg'],
        fx_px=cam_cfg['calibrated_focal_px'], fy_px=cam_cfg['calibrated_focal_px'],
        cx_px=cam_cfg['optical_center_x'], cy_px=cam_cfg['optical_center_y'],
        width=cam_cfg['width'], height=cam_cfg['height'],
        convention=convention,
    )
    print(f'LRF target -> ({u:.1f}, {v:.1f}); image center ({cam_cfg["width"]/2}, {cam_cfg["height"]/2}); behind={behind}, in_frame={in_frame}')
else:
    print('No LRF target in this frame')